## Pseudoalignment using Kallisto

Now that I know the raw data looks good, the next step is to figure out which gene/transcript each of the reads comes from. That will start painting a picture of which genes are being expressed in these cells. To do this the first step is aligning the reads to a reference.

In the actual paper they used the CellRanger pipeline for this, but that is way too intensive for my laptop. Instead I will be using Kallisto, which is much lighter and faster by doing a *pseudoalignment*. Kallisto basically takes each read and asks: "Which transcripts (mature mRNA molecules) could this have come from?"
- It does not align to precise locations in a reference genome (like CellRanger). Instead it compares reads to a collection of all transcript sequences (the transcriptome)
- It still results in a quantified expression matrix matching genes to cell counts, but trades off some extra information for things like RNA splicing. For my analysis here that is not a problem.

After this step I will download results from the actual paper at the same step to compare for the rest of my analysis.



### Conda environment

The first step is a new Conda environment with the next set of tools:

In [ ]:
%%bash

conda create -n kallisto_env -c conda-forge -c bioconda kallisto bustools
# kallisto: Pseudoalignment tool
# bustools: Works with BUS files that Kallisto will generate

conda activate kallisto_env

#### The Index

The first thing that Kallisto needs is a reference transcriptome to line up the reads to, which comes in FASTA format. In the supplementary info section of the paper the reference genome is given:
- gex-mm10-2020-A, 10X Genomics

On the 10x Genomics website we can see that this version comes from a mouse reference genome assembly called mm10 or GRCm38, and its annotations for genes and transcripts are the version GENCODE vM23:
- https://www.10xgenomics.com/support/software/cell-ranger/latest/release-notes/cr-reference-release-notes 

So with this information I can now grab a reference transcriptome from GENCODE whose set of genes exactly matches the reference genome used in CellRanger:
- Mouse - Release M23 (GRCm38.p6) - Transcript sequences (FASTA format)
- Downloaded from https://www.gencodegenes.org/mouse/release_M23.html 

Kallisto makes its index using this command:

In [ ]:
%%bash

kallisto index -i mm10.idx gencode.vM23.transcripts.fa.gz
# -i: Name for the new index

### Pseudoalignment

With this index of transcripts to compare the reads to, a pseudoalignment can now be performed. Since this is single-cell data the "kallisto bus" command is needed, resulting in BUS files that keep track of each molecule's identity.

An important flag here is "-x 10xv3", which shows the chemistry used for the initial DNA library prep. The 10x sequencing platform has had different versions with different read files and barcodes/identifiers for each molecule. I found the correct option in the paper's supplementary information section:
- "Single-cell RNA sequencing was performed using 10X Genomics NextGEM v3.1 3' Gene Expression kits."

In [ ]:
%%bash

kallisto bus -i mm10.idx -o kallisto_output -x 10xv3 -t 4 SRR36691286_1.fastq.gz SRR36691286_2.fastq.gz
# -i mm10.idx: Index to use for pseudoalignment
# -o kallisto_output: Name for the output folder
# -x 10xv3: Chemistry of the 10x library used
# -t 4: Number of threads/processors to use
# SRR36691286_1.fastq.gz SRR36691286_2.fastq.gz: Raw sequencing reads

So what does the output look like? Pseudoalignment created a BUS file (/kallisto_output/output.bus) where every molecule has a record:
- Cell barcode - Which cell this molecule came from
- Unique molecule identifier (UMI) - Unique tag to identify each molecule (clones from PCR amplification share this)
- Transcript equivalence class - List of which transcripts it could belong to

This is a helpful set of data, and what I really want to do with it is compare the amount of expression between genes. So the next step is to build a 'gene x cell count matrix':
- Rows are the different genes
- Columns are the different cells
- Values are the UMI counts

So for each entry, you can see the number of unique molecules originating from a gene detected in a cell. 

But like always, it's important to organize and clean up the data before moving on to the next step.

### Ordering/sorting the barcodes

Next step